In [16]:
!pip install python-dotenv


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import torch
import pandas as pd

In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [19]:
from dotenv import load_dotenv
load_dotenv()
import os
access_token = os.getenv('HF_TOKEN')

In [20]:
model_name = "asjc-classification/scibert_multilabel_asjc_classifier"

In [21]:
tokernizer =  AutoTokenizer.from_pretrained(model_name,token=access_token)
model = AutoModelForCausalLM.from_pretrained(model_name,token=access_token)

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
BertLMHeadModel LOAD REPORT from: asjc-classification/scibert_multilabel_asjc_classifier
Key                                        | Status     | 
-------------------------------------------+------------+-
classifier.bias                            | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | MISSING    | 
cls.predictions.transform.LayerNorm.bias   | MISSING    | 
cls.predictions.decoder.bias               | MISSING    | 
cls.predictions.transform.dense.bias       | MISSING    | 
cls.predictions.bias                       | MISSING    | 
cls.predictio

In [22]:
tokernizer.save_pretrained(f"/filed_classification_models_for_software_project/tokenizer/{model_name}")
model.save_pretrained(f"/filed_classification_models_for_software_project/model/{model_name}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
# Load merged file
df = pd.read_csv('D:/SoftwareProject/3_fileds_using_api/3filds/final_dataset.csv')

# Combine Title and Abstract for the model to read
df['input_text'] = df['title'].fillna('') + ". " + df['abstract'].fillna('')

print(f"Loaded {len(df)} rows of research metadata.")

Loaded 24015 rows of research metadata.


In [24]:
df.head()

,title,abstract,field_1_na,field_1_co,field_2_na,field_2_co,field_3_na,field_3_co,source_file,input_text
0,Discontinuities between maternity and child an...,BackgroundContinuity in the context of healthc...,"Public Health, Environmental and Occupational ...",45.00%,Health (social science),35.00%,Nursing (miscellaneous),20.00%,output2.csv,Discontinuities between maternity and child an...
1,An interactive decision-making framework (i-DM...,Background Low numbers of women in Queensland ...,Health Policy,45.00%,"Public Health, Environmental and Occupational ...",35.00%,Ecology,20.00%,output2.csv,An interactive decision-making framework (i-DM...
2,Midwifery-led care can lower caesarean section...,INTRODUCTION Midwifery-led care is recognised ...,Ecology,85.00%,"Public Health, Environmental and Occupational ...",10.00%,Maternity and Midwifery,5.00%,output2.csv,Midwifery-led care can lower caesarean section...
3,The association between midwifery staffing lev...,Background Women have consistently reported lo...,Nursing (miscellaneous),70.00%,"Public Health, Environmental and Occupational ...",20.00%,Health Policy,10.00%,output2.csv,The association between midwifery staffing lev...
4,Competencies for respectful maternity care: Id...,"BACKGROUND\nA respectful, person-centered phil...",Nursing (miscellaneous),70.00%,"Public Health, Environmental and Occupational ...",20.00%,Ecology,10.00%,output2.csv,Competencies for respectful maternity care: Id...


In [25]:
df = df.drop('source_file', axis=1)

In [26]:
df.head()

,title,abstract,field_1_na,field_1_co,field_2_na,field_2_co,field_3_na,field_3_co,input_text
0,Discontinuities between maternity and child an...,BackgroundContinuity in the context of healthc...,"Public Health, Environmental and Occupational ...",45.00%,Health (social science),35.00%,Nursing (miscellaneous),20.00%,Discontinuities between maternity and child an...
1,An interactive decision-making framework (i-DM...,Background Low numbers of women in Queensland ...,Health Policy,45.00%,"Public Health, Environmental and Occupational ...",35.00%,Ecology,20.00%,An interactive decision-making framework (i-DM...
2,Midwifery-led care can lower caesarean section...,INTRODUCTION Midwifery-led care is recognised ...,Ecology,85.00%,"Public Health, Environmental and Occupational ...",10.00%,Maternity and Midwifery,5.00%,Midwifery-led care can lower caesarean section...
3,The association between midwifery staffing lev...,Background Women have consistently reported lo...,Nursing (miscellaneous),70.00%,"Public Health, Environmental and Occupational ...",20.00%,Health Policy,10.00%,The association between midwifery staffing lev...
4,Competencies for respectful maternity care: Id...,"BACKGROUND\nA respectful, person-centered phil...",Nursing (miscellaneous),70.00%,"Public Health, Environmental and Occupational ...",20.00%,Ecology,10.00%,Competencies for respectful maternity care: Id...


In [27]:
total_unique = pd.concat([df['field_1_na'], df['field_2_na'], df['field_3_na']]).nunique()
print(f"Total unique research fields across all columns: {total_unique}")

Total unique research fields across all columns: 431


In [28]:
# Combine the three field columns into one long Series
all_fields_combined = pd.concat([df['field_1_na'], df['field_2_na'], df['field_3_na']])

# Get unique values and convert to a sorted list for consistency
unique_fields_list = sorted(all_fields_combined.dropna().unique().tolist())


print(f"Total Unique Fields: {len(unique_fields_list)}")

Total Unique Fields: 431


In [29]:
print(unique_fields_list)

['Accounting', 'Acoustics and Ultrasonics', 'Aerospace Engineering', 'Aging', 'Agricultural and Biological Sciences (miscellaneous)', 'Agronomy and Crop Science', 'Algebra and Number Theory', 'Algebraic Topology,', 'Allergy,', 'Analysis', 'Analytical Chemistry', 'Anatomy', 'Anesthesiology and Pain Medicine', 'Animal Science and Zoology', 'Anthropology', 'Applied Mathematics', 'Applied Psychology', 'Aquaculture,', 'Aquatic Science', 'Archeology', 'Archeology (arts and humanities)', 'Architecture', 'Artificial Intelligence', 'Arts and Humanities (miscellaneous)', 'Assessment and Diagnosis', 'Astronomy and Astrophysics', 'Astrophysics and Cosmology,', 'Astrophysics of Galaxies,', 'Astrophysics,', 'Atmospheric Science', 'Atomic and Molecular Physics,', 'Atomic and Molecular Physics, and Optics', 'Audiology,', 'Automotive Engineering', 'Behavioral Neuroscience', 'Bibliometrics,', 'Biochemistry', 'Biochemistry, Genetics and Molecular Biology (miscellaneous)', 'Bioengineering', 'Bioethics,', 

In [40]:
#Combine all three field columns to get the total
# count for every field
all_fields = pd.concat([df['field_1_na']])

#Calculate the frequency of each field
field_counts = all_fields.value_counts()

#Filter the field_counts for those less than 20
low_frequency_fields = field_counts[field_counts < 20]

#Get just the names as a list
low_frequency_names = low_frequency_fields.index.tolist()


print(f"Total fields with < 20 occurrences: {len(low_frequency_names)}")
print("--- List of Names ---")
for name in low_frequency_names:
    print(name)

print(low_frequency_fields)

Total fields with < 20 occurrences: 133
--- List of Names ---
Mathematical Physics
General Social Sciences
Water Science and Technology
Literature and Literary Theory
Computational Biology,
Waste Management and Disposal
Gender Studies
Conservation
Computer Graphics and Computer-Aided Design
Automotive Engineering
Forestry
Insect Science
Building and Construction
Computational Mechanics
Medical Laboratory Technology
Structural Biology
Software
Geology
General Energy
Nursing,
Molecular Biology
Geophysics
Biochemistry, Genetics and Molecular Biology (miscellaneous)
Bioinformatics,
General Dentistry
Robotics,
Logic
General Earth and Planetary Sciences
Developmental Neuroscience
Management Information Systems
Behavioral Neuroscience
Geometry and Topology
Space and Planetary Science
Arts and Humanities (miscellaneous)
Theoretical Physics,
Colloid and Surface Chemistry
Music
General Relativity and Quantum Cosmology,
Discrete Mathematics and Combinatorics
General Veterinary
Agricultural and Bi